<a href="https://colab.research.google.com/github/GokulM8/Internship/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### Setup — rebuild the honest model and baseline from Weeks 4/5/6

Same lane (Content Refresh / Opportunity Scoring), same March→April forward label, same grouped-by-client split used in ML-08/ML-09. This notebook is self-contained so it runs top to bottom on its own.

In [1]:
%pip install -q duckdb huggingface_hub

In [2]:
from google.colab import userdata
from huggingface_hub import HfApi
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
import duckdb
import numpy as np
import pandas as pd
import os
import json
import matplotlib.pyplot as plt

HF_TOKEN = userdata.get("HF_TOKEN")

In [3]:
api = HfApi()
all_files = api.list_repo_files("FlyRank/internship-warehouse", repo_type="dataset", token=HF_TOKEN)
config_files = sorted(f for f in all_files if "fact_content_daily_performance" in f and f.endswith(".parquet"))

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql(f"""CREATE OR REPLACE SECRET hf_secret (TYPE huggingface, TOKEN '{HF_TOKEN}')""")

remote_paths = [f"hf://datasets/FlyRank/internship-warehouse/{f}" for f in config_files]
paths_sql = "[" + ", ".join(f"'{p}'" for p in remote_paths) + "]"
con.sql(f"CREATE OR REPLACE VIEW fact AS SELECT * FROM read_parquet({paths_sql})")

In [4]:
march_features = con.sql("""
    WITH march AS (
        SELECT *, CAST(strftime(report_date, '%d') AS INTEGER) AS day,
            sessions_organic + sessions_direct + sessions_referral
            + sessions_social + sessions_paid + sessions_ai AS total_sessions_row
        FROM fact
        WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
    )
    SELECT
        client_hash_id, content_hash_id,
        SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS avg_ctr,
        AVG(gsc_avg_position) AS avg_position,
        SUM(gsc_impressions) AS total_impressions,
        SUM(sessions_ai) * 1.0 / NULLIF(SUM(total_sessions_row), 0) AS ai_search_share,
        SUM(ga4_engaged_sessions) * 1.0 / NULLIF(SUM(ga4_sessions), 0) AS engagement_rate,
        SUM(gsc_clicks) AS march_total_clicks,
        SUM(CASE WHEN day <= 15 THEN gsc_clicks ELSE 0 END) AS front_half_clicks,
        SUM(CASE WHEN day > 15 THEN gsc_clicks ELSE 0 END) AS back_half_clicks
    FROM march
    GROUP BY client_hash_id, content_hash_id
""").df().fillna(0)

april_totals = con.sql("""
    SELECT client_hash_id, content_hash_id, SUM(gsc_clicks) AS april_total_clicks
    FROM fact
    WHERE report_date BETWEEN DATE '2026-04-01' AND DATE '2026-04-30'
    GROUP BY client_hash_id, content_hash_id
""").df()

joined = march_features.merge(april_totals, on=["client_hash_id", "content_hash_id"], how="inner")
joined["is_declining_forward"] = (joined["april_total_clicks"] < joined["march_total_clicks"]).astype(int)

joined["decline_magnitude"] = (joined["front_half_clicks"] - joined["back_half_clicks"]).clip(lower=0)
joined["baseline_score"] = joined["decline_magnitude"] * np.log1p(joined["total_impressions"])

print("Rows:", len(joined))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 331436


In [5]:
# Honest evaluation — same grouped split as ML-08/ML-09, kept ONLY for the metrics receipts below
feature_cols = ["avg_ctr", "avg_position", "total_impressions", "ai_search_share", "engagement_rate"]
X = joined[feature_cols].fillna(0)
y = joined["is_declining_forward"]
groups = joined["client_hash_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

eval_model = RandomForestClassifier(
    n_estimators=300, max_depth=5, min_samples_leaf=10, class_weight="balanced", random_state=42,
)
eval_model.fit(X_train, y_train)
test_proba = eval_model.predict_proba(X_test)[:, 1]
held_out_auc = roc_auc_score(y_test, test_proba)

def precision_at_k(probas, labels, k=10):
    order = np.argsort(-probas)[:k]
    return labels.values[order].mean()

held_out_precision_at_10 = precision_at_k(test_proba, y_test, k=10)

print(f"Held-out ROC-AUC: {held_out_auc:.3f}")
print(f"Held-out Precision@10: {held_out_precision_at_10:.3f}")

Held-out ROC-AUC: 0.953
Held-out Precision@10: 1.000


In [6]:
# Final scoring model — refit on ALL labeled rows for the live queue.
# The number above (from the held-out split) is what goes in the metrics JSON as the honest receipt;
# this refit model is what actually scores every page for the playbook.
final_model = RandomForestClassifier(
    n_estimators=300, max_depth=5, min_samples_leaf=10, class_weight="balanced", random_state=42,
)
final_model.fit(X, y)
joined["decline_probability"] = final_model.predict_proba(X)[:, 1]

### Archetypes → actions, and the decay/refresh insight

**The decay/refresh insight (Weeks 4 & 6):** decline only matters when there's real traffic behind it — a drop on a near-zero-traffic page isn't worth a human's time, which is why both the baseline and this queue weight decline by volume rather than treating every drop the same. Separately, the CTR-vs-position signal check confirmed CTR does fall as position worsens — so a page holding a *strong* position but posting CTR below its bucket's typical rate is a distinct, different problem (a title/meta issue, not a content-decay issue) and gets a different action below.

**Four archetypes, one action and reason code each:**

| Archetype | Action | Reason code |
|---|---|---|
| Low-Volume / Low-Confidence | `MONITOR_ONLY` | `LOW_VOLUME_LOW_CONFIDENCE` |
| Declining / High-Volume | `REFRESH_CONTENT` | `DECLINE_RISK_HIGH_VOLUME` |
| Good Position / CTR Gap | `OPTIMIZE_TITLE_META` | `CTR_GAP_GOOD_POSITION` |
| Stable / Performing | `NO_ACTION` | `STABLE_PERFORMANCE` |

Low-Volume takes priority over everything else — below a certain traffic floor, none of the other signals are trustworthy enough to act on.

In [9]:
IMPRESSION_FLOOR = joined["total_impressions"].quantile(0.25)
DECLINE_THRESHOLD = 0.6

# Vectorized position bucketing (pd.cut) instead of row-wise .apply
joined["position_bucket"] = pd.cut(
    joined["avg_position"],
    bins=[-np.inf, 3, 10, 20, np.inf],
    labels=["1-3", "4-10", "11-20", "21+"],
)

# Vectorized CTR-gap check (groupby().transform) instead of row-wise .apply with a dict lookup
bucket_median_ctr = joined.groupby("position_bucket", observed=True)["avg_ctr"].transform("median")
joined["ctr_gap"] = joined["avg_ctr"] < bucket_median_ctr

# Vectorized archetype assignment (np.select) instead of row-wise .apply
conditions = [
    joined["total_impressions"] < IMPRESSION_FLOOR,
    joined["decline_probability"] >= DECLINE_THRESHOLD,
    joined["ctr_gap"] & joined["position_bucket"].isin(["1-3", "4-10"]),
]
choices = ["Low-Volume / Low-Confidence", "Declining / High-Volume", "Good Position / CTR Gap"]
joined["archetype"] = np.select(conditions, choices, default="Stable / Performing")

ARCHETYPE_ACTION = {
    "Low-Volume / Low-Confidence": ("MONITOR_ONLY", "LOW_VOLUME_LOW_CONFIDENCE", 3),
    "Declining / High-Volume": ("REFRESH_CONTENT", "DECLINE_RISK_HIGH_VOLUME", 0),
    "Good Position / CTR Gap": ("OPTIMIZE_TITLE_META", "CTR_GAP_GOOD_POSITION", 1),
    "Stable / Performing": ("NO_ACTION", "STABLE_PERFORMANCE", 2),
}

joined["action_label"] = joined["archetype"].map(lambda a: ARCHETYPE_ACTION[a][0])
joined["reason_code"] = joined["archetype"].map(lambda a: ARCHETYPE_ACTION[a][1])
joined["archetype_priority"] = joined["archetype"].map(lambda a: ARCHETYPE_ACTION[a][2])

In [10]:
# Vectorized confidence note — string ops on whole columns instead of a Python function per row
median_impressions = joined["total_impressions"].median()
vol_tier = np.where(joined["total_impressions"] >= median_impressions, "high", "low")
certainty = np.where((joined["decline_probability"] - 0.5).abs() > 0.25, "high", "moderate")

joined["confidence_note"] = (
    pd.Series(vol_tier, index=joined.index) + "-volume page, "
    + pd.Series(certainty, index=joined.index) + "-certainty model read (p="
    + joined["decline_probability"].round(2).astype(str) + ")"
)

ranked_queue = joined.sort_values(
    ["archetype_priority", "decline_probability"], ascending=[True, False]
).reset_index(drop=True)
ranked_queue.insert(0, "rank", ranked_queue.index + 1)

output_cols = [
    "rank", "client_hash_id", "content_hash_id", "archetype", "action_label", "reason_code",
    "decline_probability", "baseline_score", "confidence_note",
    "total_impressions", "avg_position", "avg_ctr",
]
ranked_queue[output_cols].head(15)

,rank,client_hash_id,content_hash_id,archetype,action_label,reason_code,decline_probability,baseline_score,confidence_note,total_impressions,avg_position,avg_ctr
0,1,client_3ffa76342f366962,content_24ee6c545832da5d,Declining / High-Volume,REFRESH_CONTENT,DECLINE_RISK_HIGH_VOLUME,0.963407,3.135494,"high-volume page, high-certainty model read (p...",22.0,3.230769,0.227273
1,2,client_3ffa76342f366962,content_8ceef2f7a067367a,Declining / High-Volume,REFRESH_CONTENT,DECLINE_RISK_HIGH_VOLUME,0.963343,3.091042,"high-volume page, high-certainty model read (p...",21.0,3.386364,0.142857
2,3,client_3ffa76342f366962,content_50ec7cb6e25d3a27,Declining / High-Volume,REFRESH_CONTENT,DECLINE_RISK_HIGH_VOLUME,0.963240,0.000000,"high-volume page, high-certainty model read (p...",19.0,2.698718,0.210526
3,4,client_f623b01661d4bfe4,content_c4bb513d1953eb52,Declining / High-Volume,REFRESH_CONTENT,DECLINE_RISK_HIGH_VOLUME,0.963081,9.133567,"high-volume page, high-certainty model read (p...",20.0,19.523810,0.150000
4,5,client_3ffa76342f366962,content_06dbdd507342156e,Declining / High-Volume,REFRESH_CONTENT,DECLINE_RISK_HIGH_VOLUME,0.962781,0.000000,"high-volume page, high-certainty model read (p...",23.0,5.533333,0.086957
5,6,client_3ffa76342f366962,content_0a8e1b2dd5b5e8ff,Declining / High-Volume,REFRESH_CONTENT,DECLINE_RISK_HIGH_VOLUME,0.962648,0.000000,"high-volume page, high-certainty model read (p...",19.0,1.777778,0.105263
6,7,client_3ffa76342f366962,content_09279f1f04924375,Declining / High-Volume,REFRESH_CONTENT,DECLINE_RISK_HIGH_VOLUME,0.962300,0.000000,"high-volume page, high-certainty model read (p...",23.0,2.770833,0.086957
7,8,client_3ffa76342f366962,content_6a674b551a1216c5,Declining / High-Volume,REFRESH_CONTENT,DECLINE_RISK_HIGH_VOLUME,0.962175,0.000000,"high-volume page, high-certainty model read (p...",6.0,4.900000,0.166667
8,9,client_3ffa76342f366962,content_2c739a6d22b87746,Declining / High-Volume,REFRESH_CONTENT,DECLINE_RISK_HIGH_VOLUME,0.961971,0.000000,"high-volume page, high-certainty model read (p...",6.0,5.333333,0.166667
9,10,client_3ffa76342f366962,content_85e9039bb1b7c4e0,Declining / High-Volume,REFRESH_CONTENT,DECLINE_RISK_HIGH_VOLUME,0.961853,0.000000,"high-volume page, high-certainty model read (p...",21.0,16.673077,0.095238


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Intended use:** a weekly/monthly prioritization aid for a content strategist deciding which pages to look at first — not an auto-publish or auto-edit pipeline. It ranks by predicted risk and observed CTR gaps; a human still makes and executes every actual edit.

**Limits:**
- Trained and evaluated on one lane (Content Refresh / Opportunity Scoring), one client slice, and one March→April window — not validated across other months, seasons, or lanes.
- The model has never seen a Google core algorithm update in its training window; a core update could break the decline signal entirely for a period.
- `decline_probability` is a directional risk read, not a guarantee — the ML-09 audit showed how much a bad split alone can move a number like this.
- Low-volume pages are explicitly downgraded to `MONITOR_ONLY`, not because they don't matter, but because the signal-to-noise ratio there is too low to trust an automated call.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Before acting on any `REFRESH_CONTENT` or `OPTIMIZE_TITLE_META` row, a human must check:**
- Has this page already been refreshed since March? (the data can lag behind reality.)
- Is the decline explained by something outside content quality — a seasonal keyword, a SERP feature change, a competitor event?
- Does the page still reflect current, accurate, compliant information?
- Is the CTR gap real, or an artifact of a title/meta test already running on that page?

**No-go — must never be automated:**
- Auto-publishing or auto-rewriting content without a human editing pass.
- Auto-deindexing, redirecting, or deleting any page based on this score alone.
- Acting on a single month of data without checking at least one prior period for consistency.
- Treating `Low-Volume / Low-Confidence` rows as if they carried the same confidence as the rest of the queue.
- Using this queue as the sole input for any decision affecting client spend, contracts, or reporting numbers — it's a content-prioritization tool, not a client-facing metric.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

- **Distribution drift:** if the monthly average `avg_ctr` or `avg_position` across scored pages shifts substantially from the training month (March), treat that as a staleness signal — check with a drift tool (e.g. Evidently) before trusting new scores.
- **Flagged-rate spike:** if the share of pages landing in `Declining / High-Volume` jumps sharply month over month, check the data pipeline first — a sync failure can look identical to a real traffic collapse.
- **Precision@10 check:** once the *actual* next-month outcome is known, recompute Precision@10 on last month's top-10 the way ML-08/ML-09 did. A sustained drop from `held_out_precision_at_10` above is the clearest retrain trigger.
- **Retrain cadence:** monthly, sliding the window forward by one month (train on month N, evaluate against month N+1), so the model never gets more than one month stale.
- **Known blind spot:** a Google core update or a client-side site migration should trigger an off-cycle retrain regardless of the above — none of these features can detect either on their own.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [11]:
os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

ranked_queue[output_cols].to_csv("work/outputs/content_action_playbook.csv", index=False)

metrics = {
    "lane": "Content Refresh / Opportunity Scoring",
    "label": "is_declining_forward (March -> April click decline)",
    "split": "GroupShuffleSplit by client_hash_id, test_size=0.3",
    "held_out_roc_auc": float(held_out_auc),
    "held_out_precision_at_10": float(held_out_precision_at_10),
    "archetype_counts": ranked_queue["archetype"].value_counts().to_dict(),
    "rows_scored": int(len(ranked_queue)),
}
with open("work/outputs/w07_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

fig, ax = plt.subplots(figsize=(6, 4))
ranked_queue["archetype"].value_counts().plot(kind="barh", ax=ax)
ax.set_xlabel("Number of pages")
ax.set_title("Content Action Playbook — archetype distribution")
fig.tight_layout()
fig.savefig("work/figures/archetype_distribution.png", dpi=150)
plt.close(fig)

print("Exported:")
print(" - work/outputs/content_action_playbook.csv  (queue — stays out of git)")
print(" - work/outputs/w07_metrics.json              (commit this — the receipts)")
print(" - work/figures/archetype_distribution.png    (commit this — reused in the paper)")

Exported:
 - work/outputs/content_action_playbook.csv  (queue — stays out of git)
 - work/outputs/w07_metrics.json              (commit this — the receipts)
 - work/figures/archetype_distribution.png    (commit this — reused in the paper)


## Self-check

Before you submit, confirm each line honestly:

- [-] Every section above is filled — markdown thinking AND the code that backs it
- [-] The notebook runs top to bottom with no errors (Runtime → Run all)
- [-] No client names, URLs, or private queries anywhere
- [-] My claims use careful words: observed, measured, directional, decision-support
- [-] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.